In [3]:
# ============================================
# Upright Row: Drive ZIP -> Extract -> FAST keypoints -> Rule labels -> CSV/PKL
# (학습은 포함하지 않음)
# ============================================

# 0) 구글 드라이브 연결
from google.colab import drive
drive.mount('/content/drive')

# 1) 경로 설정 (여기만 내 드라이브 경로에 맞게 수정)
ZIP_PATH   = "/content/drive/MyDrive/업라이트로우.zip"   # 드라이브에 올린 ZIP
IMG_ROOT   = "/content/drive/MyDrive/업라이트로우/images"           # 압축 풀릴 폴더
OUTPUT_DIR = "/content/drive/MyDrive/업라이트로우/outputs"          # 결과 저장 폴더
CSV_OUT    = f"{OUTPUT_DIR}/uprightrow_labels.csv"
PKL_OUT    = f"{OUTPUT_DIR}/uprightrow_dataset.pkl"

import os, sys
os.makedirs(IMG_ROOT, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 3) 설치/임포트
try:
    import cv2, mediapipe as mp, numpy as np, pandas as pd
except:
    !{sys.executable} -q -m pip install mediapipe==0.10.14 opencv-python-headless pandas
    import cv2, mediapipe as mp, numpy as np, pandas as pd

import glob, csv, math, pickle, time

# 4) 유틸 & 기준값(업라이트 로우)
mp_pose = mp.solutions.pose

def angle_3pt(a, b, c):
    try:
        ab = (a[0]-b[0], a[1]-b[1]); cb = (c[0]-b[0], c[1]-b[1])
        dot = ab[0]*cb[0] + ab[1]*cb[1]
        nab = math.hypot(*ab); ncb = math.hypot(*cb)
        cosang = max(-1., min(1., dot/(nab*ncb + 1e-9)))
        return math.degrees(math.acos(cosang))
    except:
        return float("nan")

def mid(p, q): return ((p[0]+q[0])/2.0, (p[1]+q[1])/2.0)
def valid_landmarks(lms): return (lms is not None) and (len(lms) == 33)

# ▶ 속도 최적화: 긴 변 256으로 리사이즈 (정확도 영향 거의 없음)
def fast_resize_keep_aspect(img, max_side=256):
    h, w = img.shape[:2]
    if max(h, w) <= max_side: return img
    if h >= w:
        new_h = max_side; new_w = int(w * (max_side / h))
    else:
        new_w = max_side; new_h = int(h * (max_side / w))
    return cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

# ▶ 업라이트 로우 규칙 임계치 (튜닝 가능)
TH = {
    # 1) 팔꿈치가 손목 리드: 팔꿈치가 손목보다 "위"(y 더 작음) + 최소 간격
    "elbow_leads_min_gap": 0.03,
    # 2) 견갑골 하강 유지: 귀-어깨 평균 수직거리 & 어깨-엉덩이 높이차
    "ear_shoulder_min": 0.03,
    "hip_minus_shoulder_min": 0.10,
    # 3) 바벨 상체 밀착: 손목 x가 가슴중앙 x에 가까움
    "hand_to_chest_tol": 0.08,
    # 4) 체간 좌우 기울임 억제(왼쪽/오른쪽으로 기울지 않기)
    "torso_center_tol": 0.06,
}

def evaluate_uprightrow(landmarks):
    L_EAR, R_EAR = 7, 8
    L_SHO, R_SHO = 11, 12
    L_ELB, R_ELB = 13, 14
    L_WRI, R_WRI = 15, 16
    L_HIP, R_HIP = 23, 24

    def xy(i): lm = landmarks[i]; return (lm.x, lm.y)

    l_ear, r_ear = xy(L_EAR), xy(R_EAR)
    l_sho, r_sho = xy(L_SHO), xy(R_SHO)
    l_elb, r_elb = xy(L_ELB), xy(R_ELB)
    l_wri, r_wri = xy(L_WRI), xy(R_WRI)
    l_hip, r_hip = xy(L_HIP), xy(R_HIP)

    chest = mid(l_sho, r_sho)
    hips  = mid(l_hip, r_hip)

    # 1) 팔꿈치가 손목 리드 (양팔 모두 만족 권장)
    cond_elbow_leads = (
        (l_elb[1] + TH["elbow_leads_min_gap"] < l_wri[1]) and
        (r_elb[1] + TH["elbow_leads_min_gap"] < r_wri[1])
    )

    # 2) 견갑골 하강 유지
    ear_sho_gap = ((l_ear[1]-l_sho[1]) + (r_ear[1]-r_sho[1])) / 2.0
    hip_minus_sho = ((l_hip[1]-l_sho[1]) + (r_hip[1]-r_sho[1])) / 2.0
    cond_scap_depress = (ear_sho_gap >= TH["ear_shoulder_min"]) and (hip_minus_sho >= TH["hip_minus_shoulder_min"])

    # 3) 바벨 상체 밀착 (손목 x ≈ 가슴중앙 x)
    wrists_x = (l_wri[0] + r_wri[0]) / 2.0
    cond_bar_close = abs(wrists_x - chest[0]) <= TH["hand_to_chest_tol"]

    # 4) 체간 수직 (어깨중심 x ≈ 엉덩이중심 x)
    cond_torso_vertical = abs(chest[0] - hips[0]) <= TH["torso_center_tol"]

    return {
        "팔꿈치가 손목 리드": cond_elbow_leads,
        "견갑골 하강 유지": cond_scap_depress,
        "바벨 상체 밀착": cond_bar_close,
        "체간 수직 유지": cond_torso_vertical,
    }

# 5) 이미지 수집
def list_images(root):
    exts = {".jpg",".jpeg",".png",".bmp"}
    paths = sorted(glob.glob(os.path.join(root, "**", "*.*"), recursive=True))
    return [p for p in paths if os.path.splitext(p)[1].lower() in exts]

image_paths = list_images(IMG_ROOT)
print("이미지 개수:", len(image_paths))
if not image_paths:
    raise SystemExit("❗ IMG_ROOT 경로를 확인하세요.")

# 6) 좌표 추출 + 라벨링 (빠른 설정: model_complexity=0, 리사이즈 256)
pose = mp_pose.Pose(static_image_mode=True, model_complexity=0,
                    enable_segmentation=False, min_detection_confidence=0.5)

# CSV 헤더
lm_cols = []
for i in range(33):
    lm_cols += [f"lm{i}_x", f"lm{i}_y", f"lm{i}_z", f"lm{i}_vis"]
header = ["image_path"] + lm_cols + ["cond_팔꿈치리드","cond_견갑하강","cond_바벨밀착","cond_체간수직","label"]

X, y, rows = [], [], []
good = bad = skipped = 0
t0 = time.time()

for idx, p in enumerate(image_paths, 1):
    img = cv2.imread(p)
    if img is None:
        skipped += 1; continue
    img = fast_resize_keep_aspect(img, 256)  # ★ 속도 최적화
    res = pose.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    if res.pose_landmarks is None or not valid_landmarks(res.pose_landmarks.landmark):
        skipped += 1; continue

    lms = res.pose_landmarks.landmark
    feats = []
    for lm in lms:
        feats += [lm.x, lm.y, lm.z, lm.visibility]
    X.append(feats)

    conds = evaluate_uprightrow(lms)
    c1 = int(conds["팔꿈치가 손목 리드"])
    c2 = int(conds["견갑골 하강 유지"])
    c3 = int(conds["바벨 상체 밀착"])
    c4 = int(conds["체간 수직 유지"])

    label = 1 if (c1 and c2 and c3 and c4) else 0
    y.append(label)
    good += (label == 1); bad += (label == 0)

    rows.append([p] + feats + [c1, c2, c3, c4, label])

    # 1천장마다 진행률 표시(출력 과다 방지)
    if idx % 1000 == 0:
        print(f"… {idx}/{len(image_paths)} 처리 중")

pose.close()
dt = time.time() - t0
print(f"⏱ 총 처리시간: {dt/60:.1f}분 | 이미지당 {dt/max(1,len(image_paths)):.4f}초")

# 7) 저장 (CSV/PKL)
with open(CSV_OUT, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows([header, *rows])

with open(PKL_OUT, "wb") as f:
    pickle.dump({"X": X, "y": y, "header": lm_cols}, f)

print(f"✅ 완료 — 정자세:{good}, 오자세:{bad}, 스킵:{skipped}, 총:{len(image_paths)}")
print("📄 CSV :", CSV_OUT)
print("📦 PKL :", PKL_OUT)

# 간단 분포 확인
import numpy as np
y_np = np.array(y)
print("라벨 분포 — 정자세(1):", int((y_np==1).sum()), "/ 오자세(0):", int((y_np==0).sum()))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
이미지 개수: 9964
… 1000/9964 처리 중
… 2000/9964 처리 중
… 3000/9964 처리 중
… 4000/9964 처리 중
… 5000/9964 처리 중
… 6000/9964 처리 중
… 7000/9964 처리 중
… 8000/9964 처리 중
… 9000/9964 처리 중
⏱ 총 처리시간: 16.4분 | 이미지당 0.0990초
✅ 완료 — 정자세:0, 오자세:9937, 스킵:27, 총:9964
📄 CSV : /content/drive/MyDrive/업라이트로우/outputs/uprightrow_labels.csv
📦 PKL : /content/drive/MyDrive/업라이트로우/outputs/uprightrow_dataset.pkl
라벨 분포 — 정자세(1): 0 / 오자세(0): 9937


In [4]:
# === CSV만으로 재라벨링 (MediaPipe 재실행 없음) ===
import pandas as pd, numpy as np, pickle

CSV_IN  = "/content/drive/MyDrive/업라이트로우/outputs/uprightrow_labels.csv"
CSV_OUT = "/content/drive/MyDrive/업라이트로우/outputs/uprightrow_labels_relabeled.csv"
PKL_OUT = "/content/drive/MyDrive/업라이트로우/outputs/uprightrow_dataset_relabeled.pkl"

# 완화된 임계값
TH = {
    "elbow_leads_min_gap": 0.015,
    "ear_shoulder_min": 0.015,
    "hip_minus_shoulder_min": 0.06,
    "hand_to_chest_tol": 0.12,
    "torso_center_tol": 0.10,
}

df = pd.read_csv(CSV_IN)

# 랜드마크 복원 (x,y만 필요)
def lm(i, axis):
    return df[f"lm{i}_{axis}"].values

l_ear_y, r_ear_y = lm(7,"y"), lm(8,"y")
l_sho_x, l_sho_y = lm(11,"x"), lm(11,"y")
r_sho_x, r_sho_y = lm(12,"x"), lm(12,"y")
l_elb_y, r_elb_y = lm(13,"y"), lm(14,"y")
l_wri_x, l_wri_y = lm(15,"x"), lm(15,"y")
r_wri_x, r_wri_y = lm(16,"x"), lm(16,"y")
l_hip_x, l_hip_y = lm(23,"x"), lm(23,"y")
r_hip_x, r_hip_y = lm(24,"x"), lm(24,"y")

chest_x = (l_sho_x + r_sho_x)/2
chest_y = (l_sho_y + r_sho_y)/2
hips_x  = (l_hip_x + r_hip_x)/2
hips_y  = (l_hip_y + r_hip_y)/2
wrists_x= (l_wri_x + r_wri_x)/2

# 1) 팔꿈치가 손목 리드 (y는 아래로 증가 → elbow_y < wrist_y)
c_elbow = ((l_elb_y + TH["elbow_leads_min_gap"] < l_wri_y) &
           (r_elb_y + TH["elbow_leads_min_gap"] < r_wri_y))

# 2) 견갑골 하강 유지
ear_sho_gap = ((l_ear_y - l_sho_y) + (r_ear_y - r_sho_y))/2
hip_minus_sho = ((l_hip_y - l_sho_y) + (r_hip_y - r_sho_y))/2
c_scap = (ear_sho_gap >= TH["ear_shoulder_min"]) & (hip_minus_sho >= TH["hip_minus_shoulder_min"])

# 3) 바벨 상체 밀착 (손목 x ≈ 가슴 x)
c_bar = np.abs(wrists_x - chest_x) <= TH["hand_to_chest_tol"]

# 4) 체간 수직 (어깨중심 x ≈ 엉덩이중심 x)
c_torso = np.abs(chest_x - hips_x) <= TH["torso_center_tol"]

conds = np.vstack([c_elbow, c_scap, c_bar, c_torso]).T
label_relaxed = (conds.sum(axis=1) >= 3).astype(int)  # 3/4 이상 만족

df["label_relaxed"] = label_relaxed
print("라벨 분포(완화 후) — 정자세(1):", (label_relaxed==1).sum(), "/ 오자세(0):", (label_relaxed==0).sum())

# 새 CSV/PKL 저장
df.to_csv(CSV_OUT, index=False)

lm_cols = [c for c in df.columns if c.startswith("lm")]
X = df[lm_cols].values.astype(np.float32)
y = df["label_relaxed"].values.astype(np.int64)
with open(PKL_OUT, "wb") as f:
    pickle.dump({"X": X, "y": y, "header": lm_cols}, f)

print("📄 CSV :", CSV_OUT)
print("📦 PKL :", PKL_OUT)


라벨 분포(완화 후) — 정자세(1): 7066 / 오자세(0): 2871
📄 CSV : /content/drive/MyDrive/업라이트로우/outputs/uprightrow_labels_relabeled.csv
📦 PKL : /content/drive/MyDrive/업라이트로우/outputs/uprightrow_dataset_relabeled.pkl


In [5]:
# ============================================
# Upright Row 모델 학습 (relabeled PKL 사용)
# RF / KNN / MLP 비교 → 최고 모델 저장
# ============================================

# 0) 경로
DATA_PKL  = "/content/drive/MyDrive/업라이트로우/outputs/uprightrow_dataset_relabeled.pkl"
MODEL_OUT = "/content/drive/MyDrive/업라이트로우/outputs/uprightrow_best_model.pkl"

# 1) 설치/임포트
import sys, os, pickle
import numpy as np
try:
    import joblib
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.neural_network import MLPClassifier
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
except:
    !{sys.executable} -q -m pip install scikit-learn joblib
    import joblib
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.neural_network import MLPClassifier
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

# 2) 데이터 로드
with open(DATA_PKL, "rb") as f:
    data = pickle.load(f)
X = np.array(data["X"], dtype=np.float32)   # (N, 132)
y = np.array(data["y"], dtype=np.int64)     # (N,)
print("Loaded:", X.shape, y.shape, " | 정자세 비율=", (y==1).mean().round(3))

# 3) Train/Test split (stratify 유지)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4) 모델들 학습/평가
results = []

# (a) RandomForest: 빠르고 튼튼
rf = RandomForestClassifier(
    n_estimators=400, random_state=42, n_jobs=-1,
    class_weight="balanced_subsample"
)
rf.fit(X_tr, y_tr)
pred = rf.predict(X_te)
acc = accuracy_score(y_te, pred)
print("\n[RandomForest] acc:", acc)
print(confusion_matrix(y_te, pred))
print(classification_report(y_te, pred, digits=4))
results.append(("RandomForest", rf, acc))

# (b) KNN: 스케일 필요
knn = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("knn", KNeighborsClassifier(n_neighbors=7))
])
knn.fit(X_tr, y_tr)
pred = knn.predict(X_te)
acc = accuracy_score(y_te, pred)
print("\n[KNN] acc:", acc)
print(confusion_matrix(y_te, pred))
print(classification_report(y_te, pred, digits=4))
results.append(("KNN", knn, acc))

# (c) MLP: 간단한 신경망
mlp = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("mlp", MLPClassifier(hidden_layer_sizes=(128,64), max_iter=300, random_state=42))
])
mlp.fit(X_tr, y_tr)
pred = mlp.predict(X_te)
acc = accuracy_score(y_te, pred)
print("\n[MLP] acc:", acc)
print(confusion_matrix(y_te, pred))
print(classification_report(y_te, pred, digits=4))
results.append(("MLP", mlp, acc))

# 5) 최고 모델 저장
best_name, best_model, best_acc = max(results, key=lambda x: x[2])
joblib.dump(best_model, MODEL_OUT)
print(f"\n🏆 Best: {best_name} (acc={best_acc:.4f}) → saved: {MODEL_OUT}")

# 6) (옵션) 프론트엔드용 간단 예측 함수
def predict_from_keypoints(feature_132, model_path=MODEL_OUT):
    """feature_132: shape (132,) or (1,132)"""
    m = joblib.load(model_path)
    x = np.asarray(feature_132, dtype=np.float32).reshape(1, -1)
    return int(m.predict(x)[0])

# 사용 예:
# pred = predict_from_keypoints(X_te[0]); print("pred:", pred)


Loaded: (9937, 132) (9937,)  | 정자세 비율= 0.711

[RandomForest] acc: 0.9612676056338029
[[ 540   34]
 [  43 1371]]
              precision    recall  f1-score   support

           0     0.9262    0.9408    0.9334       574
           1     0.9758    0.9696    0.9727      1414

    accuracy                         0.9613      1988
   macro avg     0.9510    0.9552    0.9531      1988
weighted avg     0.9615    0.9613    0.9614      1988


[KNN] acc: 0.9562374245472837
[[ 522   52]
 [  35 1379]]
              precision    recall  f1-score   support

           0     0.9372    0.9094    0.9231       574
           1     0.9637    0.9752    0.9694      1414

    accuracy                         0.9562      1988
   macro avg     0.9504    0.9423    0.9462      1988
weighted avg     0.9560    0.9562    0.9560      1988


[MLP] acc: 0.9134808853118712
[[ 413  161]
 [  11 1403]]
              precision    recall  f1-score   support

           0     0.9741    0.7195    0.8277       574
         